# ECB SPF — first look at the HICP panel

Loads all ECB SPF individual-forecast CSVs from `data/ecb/individual_forecasts/`,
extracts the year-on-year HICP inflation expectations, classifies each
`TARGET_PERIOD` string, and tabulates the panel structure (forecaster IDs,
horizons, coverage) before any rank-CI computation. This mirrors the
`EDA.ipynb` step on the Philly side.

To compute MSE-style rank CIs we still need a realization series — a Eurostat
HICP first-release vintage indexed by `target_period`. Once that exists, the
downstream pipeline is identical to the Philly notebooks (`select_top_forecasters`
→ `rank_ci_stepwise_pairwise` → `tau_best_pairwise`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rankci.data.ecb import (
    INDICATORS,
    load_ecb_csv,
    load_ecb_spf,
    list_survey_csvs,
    add_horizon,
    error_panel,
)
from rankci.data.panel import select_top_forecasters

DATA_DIR = '../../data/ecb/individual_forecasts'

## 1. Survey vintages on disk

In [ ]:
files = list_survey_csvs(DATA_DIR)
print(f'{len(files)} survey CSVs, {files[0].name} → {files[-1].name}')
print('Indicator sections expected in each CSV:')
for k, v in INDICATORS.items():
    print(f'  {k:9s} {v}')

## 2. Inspect one vintage

In [ ]:
one = load_ecb_csv(files[0])
print(f'{files[0].name} — {len(one)} rows')
print('Indicator counts:')
print(one.indicator.value_counts())
print('\nTarget-period kinds:')
print(one.target_kind.value_counts())
one.head()

## 3. Load HICP across all surveys

In [ ]:
hicp = load_ecb_spf(DATA_DIR, indicators=['HICP'])
hicp = add_horizon(hicp)
print(f'HICP rows: {len(hicp):,}')
print(f'Unique forecaster IDs: {hicp.forecaster_id.nunique()}')
print(f'Survey coverage: ({hicp.survey_year.min()},Q{hicp.survey_quarter.min()}) '
      f'→ ({hicp.survey_year.max()},Q{hicp.survey_quarter.max()})')
print(f'Target kinds: {dict(hicp.target_kind.value_counts())}')

## 4. Horizon distribution

Horizons are signed integer quarters from survey to target. The ECB short-run
panel has 0–7-quarter horizons; the long-term (5-year-ahead) block sits at
~18–21 quarters.

In [ ]:
h_dist = (hicp[hicp.horizon_q.notna()]
          .groupby('horizon_q').size()
          .rename('n_forecasts'))
print(h_dist.to_string())

## 5. Forecaster coverage

How many surveys does each forecaster show up in (any horizon)?

In [ ]:
surveys_per_fid = (hicp.drop_duplicates(['forecaster_id', 'survey_year', 'survey_quarter'])
                       .groupby('forecaster_id').size()
                       .sort_values(ascending=False))
print(f'Forecasters with ≥ 20 survey appearances: {(surveys_per_fid >= 20).sum()}')
print(f'Forecasters with ≥ 50 survey appearances: {(surveys_per_fid >= 50).sum()}')
print(f'Forecasters with ≥ 80 survey appearances: {(surveys_per_fid >= 80).sum()}')
print('\nTop 15 by coverage:')
print(surveys_per_fid.head(15))

## 6. What's missing to run rank CIs

To get from this long DataFrame to the wide `(target × forecaster)` loss panel
the rank-CI engines need, we have to supply *realized* HICP inflation for each
target period. Two options:

- **Eurostat HICP first releases** indexed by `target_period`: for `year` targets,
  the annual average year-on-year rate; for `month` targets, the rolling-month
  rate; for `quarter` targets, the quarterly rate. A `pd.Series` keyed by the
  same `TARGET_PERIOD` string used in the survey is the cleanest interface.
- **ECB SPF aggregate consensus** as a placeholder realization, just to exercise
  the pipeline (not a real evaluation, but useful for plumbing tests).

Once `realized` exists:
```python
X = error_panel(hicp, realized, indicator='HICP',
                target_kind='year', horizon_q=3, metric='squared')
X_panel = select_top_forecasters(X, N=8, min_obs=20)
from rankci import rank_ci_stepwise_pairwise
out = rank_ci_stepwise_pairwise(X_panel.values, alpha=0.05, B=5000, seed=42)
```
All downstream code is identical to the Philly notebooks.